# CSGO Modeling

This notebook benchmarks three modeling levels on the exported CSGO feature tables.

- **Baseline**: logistic regression on session-level engineered features
- **Medium**: gradient boosting on session-level engineered features
- **Larger**: optional small LSTM over ordered window sequences

The goal is to keep the Streamlit app lightweight while letting you train once, save models, and compare stronger offline options.

In [9]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
import joblib

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'csgo_generated'
MODELS = ROOT / 'artifacts' / 'csgo_models'
MODELS.mkdir(parents=True, exist_ok=True)

metadata = json.loads((DATA / 'metadata.json').read_text())
session_features = pd.read_csv(DATA / 'session_feature_table.csv')
classifier_windows = pd.read_csv(DATA / 'classifier_windows.csv')

ID_COLS = {'session_id', 'player_id', 'player_index', 'source_label', 'split', 'engagement_index', 'label_cheat'}
SESSION_FEATURE_COLS = [c for c in session_features.columns if c not in ID_COLS]

train_df = session_features[session_features['split'] == 'train'].copy()
val_df = session_features[session_features['split'] == 'validation'].copy()
test_df = session_features[session_features['split'] == 'test'].copy()

X_train, y_train = train_df[SESSION_FEATURE_COLS], train_df['label_cheat']
X_val, y_val = val_df[SESSION_FEATURE_COLS], val_df['label_cheat']
X_test, y_test = test_df[SESSION_FEATURE_COLS], test_df['label_cheat']


In [10]:
def metric_row(name, y_true, y_pred):
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0,
    }

def evaluate_threshold_search(name, model, X_val, y_val, X_test, y_test):
    val_prob = model.predict_proba(X_val)[:, 1]
    best_threshold, best_score = 0.5, -1e9
    for threshold in np.linspace(0.20, 0.85, 14):
        pred = (val_prob >= threshold).astype(int)
        score = balanced_accuracy_score(y_val, pred) + 0.20 * precision_score(y_val, pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= best_threshold).astype(int)
    row = metric_row(name, y_test, test_pred)
    row['threshold'] = best_threshold
    return row, test_prob


In [11]:
# Baseline model: logistic regression on session-level engineered features
baseline_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=600, class_weight='balanced', random_state=7)),
])
baseline_model.fit(X_train, y_train)
baseline_row, baseline_prob = evaluate_threshold_search('baseline_logistic', baseline_model, X_val, y_val, X_test, y_test)
joblib.dump(baseline_model, MODELS / 'baseline_logistic.joblib')
pd.DataFrame([baseline_row])


,model,accuracy,balanced_accuracy,precision,recall,f1,mcc,threshold
0,baseline_logistic,0.711905,0.710741,0.579235,0.706667,0.636637,0.407292,0.25


In [12]:
# Medium model: gradient boosting over the same exported session features
medium_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', HistGradientBoostingClassifier(
        max_depth=5,
        learning_rate=0.05,
        max_iter=220,
        min_samples_leaf=20,
        validation_fraction=0.1,
        random_state=7,
    )),
])
medium_model.fit(X_train, y_train)
medium_row, medium_prob = evaluate_threshold_search('medium_hgbt', medium_model, X_val, y_val, X_test, y_test)
joblib.dump(medium_model, MODELS / 'medium_hgbt.joblib')
pd.DataFrame([medium_row])


,model,accuracy,balanced_accuracy,precision,recall,f1,mcc,threshold
0,medium_hgbt,0.707143,0.646296,0.631068,0.433333,0.513834,0.325869,0.5


In [13]:
non_sequence_results = pd.DataFrame([baseline_row, medium_row]).sort_values('balanced_accuracy', ascending=False)
display(non_sequence_results)


,model,accuracy,balanced_accuracy,precision,recall,f1,mcc,threshold
0,baseline_logistic,0.711905,0.710741,0.579235,0.706667,0.636637,0.407292,0.25
1,medium_hgbt,0.707143,0.646296,0.631068,0.433333,0.513834,0.325869,0.50


## Optional Larger Models: Tiny, Small, and Medium LSTM

This path uses ordered window sequences from `classifier_windows.csv`.

The sequence length uses the full exported engagement-window history available per session, so the LSTMs are trained on the full causal window sequence rather than a short truncation.

The sweep trains three variants:

- `tiny_lstm`
- `small_lstm`
- `medium_lstm`

If `torch` is unstable on your machine, skip this section and stick to the baseline/medium models.

In [14]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
except Exception as exc:
    TORCH_OK = False
    print('Skipping LSTM section because torch is unavailable or unstable:', exc)

WINDOW_SEQUENCE_FEATURES = [
    'cheat_probability', 'emb_z', 'encoder_signal', 'automation_signature', 'movement_signature',
    'target_error_drop', 'error_speed_ratio_drop', 'error_improvement_ratio_rise',
    'tight_on_target_rise', 'lock_strength_rise', 'fire_on_target_shift', 'fire_coupling_shift',
    'fire_alignment_strength_rise', 'snap_power_rise', 'aim_efficiency_rise', 'entropy_drop',
    'micro_to_speed_drop', 'curvature_drop'
]


In [15]:
if TORCH_OK:
    def build_sequences(df, split_name, sequence_length):
        frame = df[df['split'] == split_name].sort_values(['session_id', 'window_index']).copy()
        sessions = []
        labels = []
        ids = []
        for session_id, group in frame.groupby('session_id'):
            seq = group[WINDOW_SEQUENCE_FEATURES].to_numpy(dtype=np.float32)
            if len(seq) >= sequence_length:
                seq = seq[-sequence_length:]
            sessions.append(seq)
            labels.append(int(group['label_cheat'].iloc[0]))
            ids.append(session_id)
        X = np.zeros((len(sessions), sequence_length, len(WINDOW_SEQUENCE_FEATURES)), dtype=np.float32)
        for i, seq in enumerate(sessions):
            X[i, -len(seq):, :] = seq
        return X, np.array(labels, dtype=np.int64), ids

    SEQUENCE_LENGTH = int(classifier_windows.groupby('session_id').size().max())
    print('Using sequence length (windows per engagement):', SEQUENCE_LENGTH)

    X_seq_train, y_seq_train, _ = build_sequences(classifier_windows, 'train', SEQUENCE_LENGTH)
    X_seq_val, y_seq_val, _ = build_sequences(classifier_windows, 'validation', SEQUENCE_LENGTH)
    X_seq_test, y_seq_test, test_ids = build_sequences(classifier_windows, 'test', SEQUENCE_LENGTH)

    scaler_mean = X_seq_train.reshape(-1, X_seq_train.shape[-1]).mean(axis=0)
    scaler_std = X_seq_train.reshape(-1, X_seq_train.shape[-1]).std(axis=0) + 1e-6
    X_seq_train = (X_seq_train - scaler_mean) / scaler_std
    X_seq_val = (X_seq_val - scaler_mean) / scaler_std
    X_seq_test = (X_seq_test - scaler_mean) / scaler_std

    class SequenceLSTM(nn.Module):
        def __init__(self, input_dim, hidden_dim=32, num_layers=1, head_dim=32, dropout=0.1):
            super().__init__()
            self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
            self.head = nn.Sequential(
                nn.Linear(hidden_dim, head_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(head_dim, 1),
            )
        def forward(self, x):
            out, _ = self.lstm(x)
            pooled = out[:, -1, :]
            return self.head(pooled).squeeze(-1)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    def train_lstm_model(model_name, hidden_dim, num_layers, head_dim, dropout, epochs=12):
        model = SequenceLSTM(
            input_dim=X_seq_train.shape[-1],
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            head_dim=head_dim,
            dropout=dropout,
        ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        pos_weight = torch.tensor([(len(y_seq_train) - y_seq_train.sum()) / max(1, y_seq_train.sum())], dtype=torch.float32, device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        train_ds = TensorDataset(torch.tensor(X_seq_train), torch.tensor(y_seq_train, dtype=torch.float32))
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

        best_state = None
        best_val = -1e9
        for epoch in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                logits = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
            model.eval()
            with torch.no_grad():
                val_logits = model(torch.tensor(X_seq_val, dtype=torch.float32, device=device))
                val_prob = torch.sigmoid(val_logits).cpu().numpy()
            val_pred = (val_prob >= 0.5).astype(int)
            val_score = balanced_accuracy_score(y_seq_val, val_pred)
            if val_score > best_val:
                best_val = val_score
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            val_logits = model(torch.tensor(X_seq_val, dtype=torch.float32, device=device))
            val_prob = torch.sigmoid(val_logits).cpu().numpy()
            test_logits = model(torch.tensor(X_seq_test, dtype=torch.float32, device=device))
            test_prob = torch.sigmoid(test_logits).cpu().numpy()

        best_threshold, best_score = 0.5, -1e9
        for threshold in np.linspace(0.20, 0.85, 14):
            pred = (val_prob >= threshold).astype(int)
            score = balanced_accuracy_score(y_seq_val, pred) + 0.20 * precision_score(y_seq_val, pred, zero_division=0)
            if score > best_score:
                best_score = score
                best_threshold = float(threshold)

        test_pred = (test_prob >= best_threshold).astype(int)
        row = metric_row(model_name, y_seq_test, test_pred)
        row['threshold'] = best_threshold
        row['sequence_length'] = SEQUENCE_LENGTH
        row['hidden_dim'] = hidden_dim
        row['num_layers'] = num_layers
        row['head_dim'] = head_dim
        row['param_count'] = int(sum(p.numel() for p in model.parameters()))

        torch.save(
            {
                'state_dict': model.state_dict(),
                'input_features': WINDOW_SEQUENCE_FEATURES,
                'scaler_mean': scaler_mean,
                'scaler_std': scaler_std,
                'threshold': best_threshold,
                'sequence_length': SEQUENCE_LENGTH,
                'hidden_dim': hidden_dim,
                'num_layers': num_layers,
                'head_dim': head_dim,
            },
            MODELS / f'{model_name}.pt',
        )
        return row

    lstm_specs = [
        ('tiny_lstm', 32, 1, 32, 0.10),
        ('small_lstm', 56, 2, 64, 0.15),
        ('medium_lstm', 80, 2, 96, 0.20),
    ]
    lstm_rows = [train_lstm_model(*spec) for spec in lstm_specs]
    display(pd.DataFrame(lstm_rows).sort_values('balanced_accuracy', ascending=False))
else:
    lstm_rows = []


Using sequence length (windows per engagement): 21


,model,accuracy,balanced_accuracy,precision,recall,f1,mcc,threshold,sequence_length,hidden_dim,num_layers,head_dim,param_count
2,medium_lstm,0.692857,0.692963,0.556150,0.693333,0.617211,0.372077,0.25,21,80,2,96,91713
0,tiny_lstm,0.738095,0.688148,0.675439,0.513333,0.583333,0.405457,0.65,21,32,1,32,7745
1,small_lstm,0.716667,0.680370,0.614815,0.553333,0.582456,0.370112,0.45,21,56,2,64,46273


In [16]:
comparison_rows = [baseline_row, medium_row]
if 'lstm_rows' in globals() and lstm_rows:
    comparison_rows.extend(lstm_rows)

comparison_df = pd.DataFrame(comparison_rows)
display_cols = [
    'model', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'mcc', 'threshold',
    'sequence_length', 'hidden_dim', 'num_layers', 'head_dim', 'param_count'
]
for col in display_cols:
    if col not in comparison_df.columns:
        comparison_df[col] = np.nan

comparison_df = comparison_df[display_cols].sort_values(
    ['balanced_accuracy', 'f1', 'precision'], ascending=False
).reset_index(drop=True)
display(comparison_df)


,model,accuracy,balanced_accuracy,precision,recall,f1,mcc,threshold,sequence_length,hidden_dim,num_layers,head_dim,param_count
0,baseline_logistic,0.711905,0.710741,0.579235,0.706667,0.636637,0.407292,0.25,NaN,NaN,NaN,NaN,NaN
1,medium_lstm,0.692857,0.692963,0.556150,0.693333,0.617211,0.372077,0.25,21.0,80.0,2.0,96.0,91713.0
2,tiny_lstm,0.738095,0.688148,0.675439,0.513333,0.583333,0.405457,0.65,21.0,32.0,1.0,32.0,7745.0
3,small_lstm,0.716667,0.680370,0.614815,0.553333,0.582456,0.370112,0.45,21.0,56.0,2.0,64.0,46273.0
4,medium_hgbt,0.707143,0.646296,0.631068,0.433333,0.513834,0.325869,0.50,NaN,NaN,NaN,NaN,NaN
